# Mixture of Experts: Scaling Transformers with Conditional Computation

This notebook implements **Mixture of Experts (MoE)** for Transformers — the technique
behind Mixtral 8x7B, GPT-4, and DeepSeek-V2. MoE scales model capacity (more parameters)
without proportionally increasing compute: each token is routed to only a few "expert"
sub-networks out of many.

We will:
1. Understand why dense scaling hits a wall and how MoE breaks through it
2. Implement a top-K router (gating network) from scratch
3. Build a load balancing loss to prevent expert collapse
4. Create a full MoE Transformer block (drop-in FFN replacement)
5. Compare Switch Transformer (K=1) vs Mixtral (K=2) routing
6. Visualize expert routing patterns

**References:**
- Shazeer et al. (2017). *Outrageously Large Neural Networks: The Sparsely-Gated MoE Layer.* https://arxiv.org/abs/1701.06538
- Fedus et al. (2022). *Switch Transformers: Scaling to Trillion Parameter Models.* https://arxiv.org/abs/2101.03961
- Jiang et al. (2024). *Mixtral of Experts.* https://arxiv.org/abs/2401.04088

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
from src.utils.device import set_seed

set_seed(42)
print(f"PyTorch version: {torch.__version__}")

## 1. The Scaling Problem

In a dense Transformer, **every token uses every parameter** in every layer.
Doubling parameters = doubling compute per token.

MoE breaks this relationship:

| Model | Total Params | Active Params/Token | Compute |
|---|---|---|---|
| LLaMA 2 7B (dense) | 7B | 7B (100%) | 1x |
| LLaMA 2 13B (dense) | 13B | 13B (100%) | ~2x |
| **Mixtral 8x7B (MoE)** | **47B** | **~13B (28%)** | **~2x** |

Mixtral has **47 billion** parameters but only activates ~13B per token — comparable
compute to a 13B dense model, but with the knowledge capacity of a much larger one.

## 2. The MoE Idea

Replace the **single FFN** in each Transformer block with **N expert FFNs** + a **router**
that selects the top-K experts for each token:

```
Dense Transformer Block:          MoE Transformer Block:

  token → Attention → FFN → out     token → Attention → Router → Expert 1 → out
          (all params used)                              ↘ Expert 2 ↗
                                                          Expert 3 (unused)
                                                          Expert 4 (unused)
                                                          ...
                                                          Expert N (unused)
```

Each expert is a **standard FFN** (same architecture, different weights). The router is
a simple linear layer that decides which K experts to activate for each token.

**Conditional computation:** more capacity (N experts worth of parameters) without
proportionally more compute (only K experts run per token).

In [ ]:
# Visualize: dense vs MoE compute
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Dense: all params used
ax = axes[0]
colors = ['tab:blue'] * 8
ax.barh(range(8), [1]*8, color=colors, alpha=0.8, edgecolor='black')
ax.set_yticks(range(8))
ax.set_yticklabels([f'Param block {i}' for i in range(8)])
ax.set_xlabel('Active', fontsize=12)
ax.set_title('Dense FFN: ALL params used per token', fontsize=13)
ax.set_xlim(0, 1.2)

# MoE: only K of N used
ax = axes[1]
active = [True, False, True, False, False, False, False, False]  # top-2 of 8
colors = ['tab:green' if a else 'lightgray' for a in active]
ax.barh(range(8), [1]*8, color=colors, alpha=0.8, edgecolor='black')
ax.set_yticks(range(8))
ax.set_yticklabels([f'Expert {i}' for i in range(8)])
ax.set_xlabel('Active', fontsize=12)
ax.set_title('MoE FFN: Only 2 of 8 experts per token', fontsize=13)
ax.set_xlim(0, 1.2)

plt.tight_layout()
plt.show()

# Parameter comparison
d_model, d_ff, n_experts, top_k = 4096, 14336, 8, 2
dense_params = 2 * d_model * d_ff  # one FFN
moe_params = n_experts * 2 * d_model * d_ff  # N expert FFNs
moe_active = top_k * 2 * d_model * d_ff  # only K active
print(f"FFN params per layer (Mixtral-scale):")
print(f"  Dense:      {dense_params/1e6:.0f}M (all active)")
print(f"  MoE total:  {moe_params/1e6:.0f}M ({n_experts} experts)")
print(f"  MoE active: {moe_active/1e6:.0f}M (top-{top_k} of {n_experts})")

## 3. The Router (Gating Network)

The router is surprisingly simple — just a linear projection:

$G(x) = \text{Softmax}(\text{TopK}(x \cdot W_g))$

where $W_g \in \mathbb{R}^{d_{model} \times N}$ maps each token to a score for each expert.

**Top-K selection:** Keep only the K highest scores, set the rest to $-\infty$ before softmax.
This ensures exactly K experts are activated per token.

- **Switch Transformer:** $K = 1$ (simplest, each token → 1 expert)
- **Mixtral:** $K = 2$ (each token → 2 experts, outputs weighted by router scores)

In [ ]:
class Router(nn.Module):
    """Top-K router for Mixture of Experts."""

    def __init__(self, d_model, n_experts, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.gate = nn.Linear(d_model, n_experts, bias=False)

    def forward(self, x):
        """
        Args:
            x: [batch, seq_len, d_model]
        Returns:
            weights: [batch, seq_len, top_k] — softmax weights for selected experts
            indices: [batch, seq_len, top_k] — which experts were selected
            router_logits: [batch, seq_len, n_experts] — raw scores (for aux loss)
        """
        router_logits = self.gate(x)  # [batch, seq, n_experts]

        # Select top-K experts
        top_k_logits, indices = torch.topk(router_logits, self.top_k, dim=-1)

        # Softmax over selected experts only
        weights = torch.softmax(top_k_logits, dim=-1)

        return weights, indices, router_logits


# Demo: route 8 tokens to 2 of 4 experts
router = Router(d_model=16, n_experts=4, top_k=2)
x = torch.randn(1, 8, 16)  # 8 tokens
weights, indices, logits = router(x)

print(f"Input: {x.shape} (8 tokens)")
print(f"Weights: {weights.shape} (top-2 softmax weights per token)")
print(f"Indices: {indices.shape} (which 2 experts per token)")
print(f"\nRouting for each token:")
for t in range(8):
    exp = indices[0, t].tolist()
    w = weights[0, t].detach().numpy()
    print(f"  Token {t}: experts {exp}, weights [{w[0]:.2f}, {w[1]:.2f}]")

## 4. Load Balancing

**Problem:** Without intervention, the router learns to always pick the same expert(s).
Most experts go unused ("expert collapse"), wasting parameters.

**Solution:** An auxiliary loss that encourages uniform routing:

$L_{aux} = \alpha \cdot N \sum_{i=1}^{N} f_i \cdot P_i$

where:
- $f_i$ = fraction of tokens routed to expert $i$ (actual load)
- $P_i$ = average routing probability for expert $i$ (intended load)
- $N$ = number of experts
- $\alpha$ = loss coefficient (typically 0.01)

This loss is minimized when all experts receive equal traffic ($f_i = P_i = 1/N$).

In [ ]:
def load_balancing_loss(router_logits, top_k_indices, n_experts, alpha=0.01):
    """
    Auxiliary loss for load balancing across experts.
    
    Args:
        router_logits: [batch, seq, n_experts] raw router scores
        top_k_indices: [batch, seq, top_k] selected expert indices
        n_experts: total number of experts
        alpha: loss coefficient
    """
    # f_i: fraction of tokens routed to each expert
    flat_indices = top_k_indices.reshape(-1)  # all selected experts
    expert_counts = torch.zeros(n_experts)
    for i in range(n_experts):
        expert_counts[i] = (flat_indices == i).float().sum()
    f = expert_counts / flat_indices.numel()  # normalize

    # P_i: average routing probability for each expert
    probs = torch.softmax(router_logits, dim=-1)  # [batch, seq, n_experts]
    P = probs.mean(dim=(0, 1))  # average over batch and seq

    # Auxiliary loss
    loss = alpha * n_experts * (f * P).sum()
    return loss, f, P


# Demo: compare balanced vs imbalanced routing
n_exp = 4

# Balanced: each expert gets ~25% of tokens
balanced_logits = torch.randn(1, 100, n_exp) * 0.1  # low variance → uniform
_, balanced_idx, _ = Router(16, n_exp, 2)(torch.randn(1, 100, 16))

loss_b, f_b, P_b = load_balancing_loss(balanced_logits, balanced_idx, n_exp)

# Imbalanced: one expert gets most tokens
imbalanced_logits = torch.zeros(1, 100, n_exp)
imbalanced_logits[:, :, 0] = 10.0  # expert 0 dominates
imbalanced_idx = torch.zeros(1, 100, 2, dtype=torch.long)  # all go to expert 0

loss_i, f_i, P_i = load_balancing_loss(imbalanced_logits, imbalanced_idx, n_exp)

print("Balanced routing:")
print(f"  Load per expert: {f_b.numpy().round(3)}")
print(f"  Aux loss: {loss_b.item():.4f}")
print(f"\nImbalanced routing (expert 0 dominates):")
print(f"  Load per expert: {f_i.numpy().round(3)}")
print(f"  Aux loss: {loss_i.item():.4f}")
print(f"\n→ Higher loss = more imbalanced = optimizer pushes toward balance")

## 5. MoE FFN Layer

The MoE layer replaces the single FFN with N expert FFNs + router. Each expert is a
standard 2-layer MLP. The output for each token is a **weighted sum** of its selected
expert outputs:

$y = \sum_{i \in \text{TopK}} w_i \cdot \text{Expert}_i(x)$

where $w_i$ are the router softmax weights for the selected experts.

In [ ]:
class Expert(nn.Module):
    """A single expert: standard FFN (same as in a dense Transformer)."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w2(self.dropout(F.relu(self.w1(x))))


class MoELayer(nn.Module):
    """Mixture of Experts layer: router + N expert FFNs."""

    def __init__(self, d_model, d_ff, n_experts=8, top_k=2, dropout=0.1):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.router = Router(d_model, n_experts, top_k)
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff, dropout) for _ in range(n_experts)
        ])

    def forward(self, x):
        """
        Args:
            x: [batch, seq_len, d_model]
        Returns:
            output: [batch, seq_len, d_model]
            router_logits: for auxiliary loss
        """
        batch, seq_len, d = x.shape
        weights, indices, router_logits = self.router(x)

        # Compute expert outputs and combine
        output = torch.zeros_like(x)
        for k in range(self.top_k):
            expert_idx = indices[:, :, k]  # [batch, seq]
            expert_weight = weights[:, :, k].unsqueeze(-1)  # [batch, seq, 1]

            for i in range(self.n_experts):
                mask = (expert_idx == i)  # which tokens go to expert i
                if mask.any():
                    expert_input = x[mask]  # gather tokens for this expert
                    expert_output = self.experts[i](expert_input)
                    # Scatter back, weighted by router score
                    output[mask] += expert_weight[mask].squeeze(-1).unsqueeze(-1) * expert_output

        return output, router_logits


# Demo
moe = MoELayer(d_model=32, d_ff=64, n_experts=4, top_k=2)
x = torch.randn(1, 8, 32)
out, logits = moe(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Router logits: {logits.shape} (for load balancing loss)")

In [ ]:
class MoETransformerBlock(nn.Module):
    """Transformer block with MoE FFN (drop-in replacement for dense block)."""

    def __init__(self, d_model, n_heads, d_ff, n_experts=8, top_k=2, dropout=0.1):
        super().__init__()
        # Standard self-attention
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        # MoE FFN instead of dense FFN
        self.moe = MoELayer(d_model, d_ff, n_experts, top_k, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + residual + norm
        attn_out, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.dropout(attn_out))
        # MoE FFN + residual + norm
        moe_out, router_logits = self.moe(x)
        x = self.norm2(x + self.dropout(moe_out))
        return x, router_logits


block = MoETransformerBlock(d_model=32, n_heads=4, d_ff=64, n_experts=4, top_k=2)
x = torch.randn(1, 8, 32)
out, logits = block(x)
print(f"MoE Transformer Block: {x.shape} → {out.shape}")

In [ ]:
# Compare parameter counts: dense vs MoE
d_model, n_heads, d_ff = 64, 4, 256
n_experts, top_k = 8, 2

# Dense block (standard FFN)
dense_ffn_params = 2 * d_model * d_ff + d_model + d_ff  # w1, w2, biases

# MoE block (N expert FFNs + router)
moe_ffn_params = n_experts * dense_ffn_params + d_model * n_experts  # experts + router
moe_active_params = top_k * dense_ffn_params  # only K experts active per token

print(f"Parameter comparison (d={d_model}, d_ff={d_ff}):")
print(f"  Dense FFN:     {dense_ffn_params:>8,} params (all active per token)")
print(f"  MoE FFN total: {moe_ffn_params:>8,} params ({n_experts} experts)")
print(f"  MoE active:    {moe_active_params:>8,} params (top-{top_k} per token)")
print(f"\n  MoE has {moe_ffn_params/dense_ffn_params:.1f}x more params")
print(f"  But only {moe_active_params/dense_ffn_params:.1f}x compute per token")

In [ ]:
# Visualize expert routing patterns
moe_viz = MoELayer(d_model=32, d_ff=64, n_experts=8, top_k=2)
x = torch.randn(1, 16, 32)  # 16 tokens
_, _, router_logits = moe_viz.router(x)

# Get routing decisions
_, indices = torch.topk(router_logits[0], 2, dim=-1)  # [16, 2]

# Build routing matrix
routing = torch.zeros(16, 8)
for t in range(16):
    for k in range(2):
        routing[t, indices[t, k]] = 1

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(routing.numpy().T, cmap='Greens', aspect='auto', vmin=0, vmax=1)
ax.set_xlabel('Token', fontsize=12)
ax.set_ylabel('Expert', fontsize=12)
ax.set_title('Expert Routing Pattern (top-2 of 8 experts, 16 tokens)', fontsize=14)
ax.set_yticks(range(8))
ax.set_yticklabels([f'Expert {i}' for i in range(8)])
plt.colorbar(im, ax=ax, label='Active')
plt.tight_layout()
plt.show()

# Print load distribution
load = routing.sum(dim=0) / routing.sum()
print(f"Load distribution: {load.numpy().round(3)}")
print(f"Ideal (uniform): {1/8:.3f} per expert")

## 6. Switch Transformer (K=1)

The simplest MoE: each token goes to **exactly one expert** (K=1).

**Pros:**
- Simplest routing logic — no weighted combination
- Less communication overhead in distributed settings
- Google showed it works at trillion-parameter scale (Switch Transformer, 2022)

**Cons:**
- Less capacity per token (only 1 expert's worth of computation)
- More sensitive to routing errors (wrong expert = bad output)
- Needs careful capacity factor management (what if expert is full?)

## 7. Mixtral (K=2)

Mixtral routes each token to **2 of 8 experts**, combining outputs with router weights:

$y = w_1 \cdot \text{Expert}_{i_1}(x) + w_2 \cdot \text{Expert}_{i_2}(x)$

**Architecture:** 8 experts, each is a standard 7B FFN. Total: 47B params.
Active per token: ~13B (2 experts + shared attention).

**Why K=2 works better than K=1:**
- Tokens get perspectives from 2 different experts
- More robust to routing mistakes (if one expert is wrong, the other can compensate)
- Smooth gradient flow through 2 paths

Mixtral proved that MoE works for **open-source LLMs** — it matches LLaMA 2 70B
quality with much less compute.

## 8. Practical Challenges

MoE introduces unique engineering challenges:

**Expert parallelism** — each expert lives on a different GPU. Tokens must be
sent to the right GPU (all-to-all communication), processed, then sent back.

**Load imbalance** — if the router sends 80% of tokens to one expert, that GPU
becomes a bottleneck while others sit idle. The auxiliary loss helps, but doesn't
fully solve this.

**Capacity factor** — if an expert's buffer is full, excess tokens are **dropped**
(their output is just the residual connection). Typical capacity factor: 1.25
(each expert can handle 25% more than its fair share).

**Expert collapse** — without load balancing, the router converges to using only
1-2 experts. The rest have untrained weights and become useless.

In [ ]:
# Build a small MoE Transformer and run a forward pass
class MoETransformer(nn.Module):
    """Small MoE Transformer for demonstration."""

    def __init__(self, vocab_size, d_model, n_heads, d_ff,
                 n_layers, n_experts, top_k, max_len=512, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([
            MoETransformerBlock(d_model, n_heads, d_ff, n_experts, top_k, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.shape[1]
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embed(x) + self.pos_embed(pos)

        all_router_logits = []
        for layer in self.layers:
            x, router_logits = layer(x)
            all_router_logits.append(router_logits)

        x = self.norm(x)
        return self.head(x), all_router_logits


# Create model with Mixtral-style config (but tiny)
model = MoETransformer(
    vocab_size=1000, d_model=64, n_heads=4, d_ff=128,
    n_layers=4, n_experts=8, top_k=2
)

tokens = torch.randint(0, 1000, (1, 16))  # 16 tokens
logits, router_logits_list = model(tokens)

total_params = sum(p.numel() for p in model.parameters())
print(f"MoE Transformer (4 layers, 8 experts, top-2):")
print(f"  Input:  {tokens.shape}")
print(f"  Output: {logits.shape}")
print(f"  Total parameters: {total_params:,}")

# Compute load balancing loss
total_aux_loss = 0
for rl in router_logits_list:
    _, idx = torch.topk(rl, 2, dim=-1)
    aux, _, _ = load_balancing_loss(rl, idx, 8)
    total_aux_loss += aux
print(f"  Aux loss (load balancing): {total_aux_loss.item():.4f}")

## 9. MoE in Production

| Model | Year | Experts | Top-K | Total Params | Active Params |
|---|---|---|---|---|---|
| Switch Transformer | 2022 | 128 | 1 | 1.6T | 12.5B |
| Mixtral 8x7B | 2024 | 8 | 2 | 47B | ~13B |
| Mixtral 8x22B | 2024 | 8 | 2 | 176B | ~44B |
| DBRX | 2024 | 16 | 4 | 132B | ~36B |
| Grok-1 | 2024 | 8 | 2 | 314B | ~79B |
| DeepSeek-V2 | 2024 | 160 | 6 | 236B | ~21B |
| GPT-4 | 2023 | ? | ? | ? | ? (rumored MoE) |

## 10. Dense vs MoE Comparison

| | Dense Transformer | MoE Transformer |
|---|---|---|
| Params per token | All | K/N fraction |
| Scaling | Linear (2x params = 2x compute) | **Sublinear** (8x params ≈ 2x compute) |
| Memory | Proportional to params | **All experts in memory** (even unused) |
| Training | Straightforward | Needs load balancing, capacity management |
| Inference | Simple | All-to-all routing, expert parallelism |
| Quality per FLOP | Baseline | **Better** (more knowledge per compute) |

In [ ]:
# Compare: tiny dense vs MoE on random data (few training steps)
# Shows that MoE has more capacity (lower loss) for similar active compute

dense_model = nn.Sequential(
    nn.Embedding(100, 32),
    nn.TransformerEncoder(
        nn.TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=128, batch_first=True),
        num_layers=2
    ),
    nn.Linear(32, 100),
)

moe_model = MoETransformer(
    vocab_size=100, d_model=32, n_heads=4, d_ff=128,
    n_layers=2, n_experts=4, top_k=1
)

dense_params = sum(p.numel() for p in dense_model.parameters())
moe_params = sum(p.numel() for p in moe_model.parameters())

# Train both for a few steps on random next-token prediction
opt_d = torch.optim.Adam(dense_model.parameters(), lr=1e-3)
opt_m = torch.optim.Adam(moe_model.parameters(), lr=1e-3)

dense_losses, moe_losses = [], []
for step in range(50):
    data = torch.randint(0, 100, (4, 16))
    target = torch.randint(0, 100, (4, 16))

    # Dense
    out_d = dense_model(data)
    loss_d = F.cross_entropy(out_d.view(-1, 100), target.view(-1))
    opt_d.zero_grad(); loss_d.backward(); opt_d.step()
    dense_losses.append(loss_d.item())

    # MoE
    out_m, rl = moe_model(data)
    loss_m = F.cross_entropy(out_m.view(-1, 100), target.view(-1))
    opt_m.zero_grad(); loss_m.backward(); opt_m.step()
    moe_losses.append(loss_m.item())

print(f"Dense: {dense_params:,} params, final loss: {dense_losses[-1]:.3f}")
print(f"MoE:   {moe_params:,} params, final loss: {moe_losses[-1]:.3f}")
print(f"MoE has {moe_params/dense_params:.1f}x more params but similar active compute")

## 11. Key Takeaways

1. **MoE scales parameters without proportionally scaling compute.** Each token uses only K of N expert FFNs. Mixtral 8x7B: 47B total params, ~13B active per token.

2. **The router is just a linear layer.** $G(x) = \text{Softmax}(\text{TopK}(x W_g))$. Simple, differentiable, and learned end-to-end.

3. **Load balancing is critical.** Without the auxiliary loss, the router collapses to always picking the same expert(s). The loss encourages uniform distribution across experts.

4. **MoE replaces the FFN, not attention.** The self-attention layer is shared across all tokens (same as dense). Only the FFN becomes conditional. This is because FFN parameters are the majority of a Transformer's params.

5. **K=2 (Mixtral) works better than K=1 (Switch) for LLMs.** Two expert perspectives per token gives more capacity and robustness, at the cost of slightly more compute.

6. **MoE has unique engineering challenges.** Expert parallelism, all-to-all communication, capacity management, and expert collapse are all practical concerns at scale.

### Related Notebooks

- Basic MoE with CNNs: `02_Architectures/CNNs/CNN_9_MOE.ipynb`
- Transformer architecture: `05_Papers/03_Attention_Is_All_You_Need.ipynb`
- Flash Attention (efficient attention): `03_Training_Techniques/01_Flash_Attention.ipynb`

### Further Reading

- Shazeer et al. (2017). *Outrageously Large Neural Networks.* https://arxiv.org/abs/1701.06538
- Fedus et al. (2022). *Switch Transformers.* https://arxiv.org/abs/2101.03961
- Jiang et al. (2024). *Mixtral of Experts.* https://arxiv.org/abs/2401.04088
- Lepikhin et al. (2020). *GShard.* https://arxiv.org/abs/2006.16668